# Additional Exception Concepts — Robust Boundaries and Recovery 🛡️

## Goal

Learn exception hierarchy, exception groups, cleanup design, retries, logging, and result-oriented alternatives.

## Hierarchy and tuples of exceptions

In [1]:
print(issubclass(FileNotFoundError,OSError),issubclass(ValueError,Exception))
try: int(None)
except (TypeError,ValueError) as error: print(type(error).__name__,error)

True True
TypeError int() argument must be a string, a bytes-like object or a real number, not 'NoneType'


## Custom context and `raise from`

In [2]:
class ConfigurationError(Exception): pass
def parse_port(text):
    try: port=int(text)
    except ValueError as error: raise ConfigurationError('port must contain digits') from error
    if not 1<=port<=65535: raise ConfigurationError('port outside 1..65535')
    return port
try: parse_port('abc')
except ConfigurationError as error: print(error,'<-',type(error.__cause__).__name__)

port must contain digits <- ValueError


## Retry only transient operations

Retries need a limit and should target temporary failures, not invalid input or programming bugs.

In [3]:
def retry(operation,attempts=3):
    last=None
    for number in range(1,attempts+1):
        try: return operation()
        except TimeoutError as error: last=error; print('retry',number)
    raise last
state={'n':0}
def sometimes():
    state['n']+=1
    if state['n']<3: raise TimeoutError('temporary')
    return 'ok'
print(retry(sometimes))

retry 1
retry 2
ok


## Result objects when failure is ordinary

Exceptions suit exceptional failure. For expected validation results, returning structured success/errors can be clearer.

In [4]:
def validate_age(text):
    try: age=int(text)
    except (TypeError,ValueError): return {'ok':False,'error':'whole number required'}
    if not 0<=age<=130: return {'ok':False,'error':'outside range'}
    return {'ok':True,'value':age}
print(validate_age('abc'),validate_age('25'))

{'ok': False, 'error': 'whole number required'} {'ok': True, 'value': 25}


## Logging and security

Log technical traceback details for maintainers; show a calm safe message to users. Avoid logging passwords, tokens, personal data, or full sensitive file contents. Python 3.11 also supports `ExceptionGroup` and `except*` for multiple independent failures.

## Key concepts summary

Robust systems add context, distinguish transient from permanent failure, and choose exceptions versus result data intentionally.

## Important syntax and quick revision cheat sheet

| Need | Syntax |
|---|---|
| Current exception | `raise` |
| Original cause | `raise NewError() from error` |
| Retry | bounded loop around specific transient error |
| Validation result | `{ok, value/error}` |

## Common mistakes and interview tips

- Do not retry deterministic invalid input.
- Preserve tracebacks with chaining.
- Keep user messages separate from technical logs.
- Catch only errors you can meaningfully handle.

**Revision habit:** explain what each line does, predict the result, run it, and test one edge case.